# Ampliación de datos: pipeline Bronze → Silver → Gold

Este notebook es el mismo pipeline de la entrega 2 (`pipeline_bronze_silver_gold.ipynb`), ejecutado sobre más categorías. En la entrega 2 trabajé con 3 categorías (arroz, legumbres y pasta, 88 productos). Para poder entrenar y validar un modelo de Machine Learning necesito más datos, así que aquí lo amplío a **43 subcategorías de alimentos envasados** de Mercadona.

**Qué se mantiene igual:**
- Las tres capas: Bronze (descarga sin tocar), Silver (limpieza, unión por código de barras y controles de calidad) y Gold (un producto por fila).
- Los controles de la entrega 2: coherencia de calorías, rangos, precios positivos, fibra 0 frente a fibra ausente, fecha real de descarga y la tabla de correcciones manuales.

**Qué cambia:**
- Una lista de categorías más larga (celda de descarga).
- Una columna `grupo` (base, snack_dulce, plato_preparado, salsa) para poder analizar los resultados por tipo de producto.
- `estado` (seco, cocinado, fresco, instantáneo) solo tiene sentido en arroz, legumbres y pasta; en el resto vale `no_aplica`.
- Una columna `precio_kg`, porque con más categorías el precio de referencia ya no es siempre por kilo.
- Los archivos de salida terminan en `_ampliado` para no pisar los de la entrega 2.

Los datos descargados y el Gold no se publican en el repositorio. El experimento de Machine Learning que usa este Gold está en `experimento_ml.ipynb`.

In [1]:
import json
import os
import time
import datetime

import numpy as np
import pandas as pd
import requests

# Open Food Facts pide identificarse con un User-Agent
headers = {"User-Agent": "TFM-DataScience/0.1 (Aroa)"}

# Carpetas de datos (se crean si no existen)
for carpeta in [
    "data/raw/mercadona/categories",
    "data/raw/mercadona/products",
    "data/raw/openfoodfacts",
    "data/silver",
    "data/gold",
]:
    os.makedirs(carpeta, exist_ok=True)

## 1. Bronze: descarga de datos en crudo

Por cada categoría de Mercadona hago tres cosas: descargo la lista de productos, descargo la ficha de cada producto (que es donde aparece su código de barras) y, con ese código, busco el producto en Open Food Facts. Todo se guarda sin modificar en la carpeta data/raw.

Tres decisiones que he tomado:
- Uno los datos de las dos fuentes por el código de barras y nunca por el nombre. Los nombres cambian de una web a otra y pueden confundirse; el código de barras es único para cada producto.
- Si un archivo ya está descargado, lo reutilizo. Así no repito peticiones y respeto a las webs.
- Open Food Facts bloquea durante un rato a quien le hace demasiadas peticiones seguidas. Cuando pasa, el código espera unos segundos y lo vuelve a intentar. Si aun así falla, el producto queda marcado en la columna off_error.

In [2]:
def pedir_off(ean, headers, intentos=3):
    url = f"https://world.openfoodfacts.org/api/v2/product/{ean}.json"
    for intento in range(intentos):
        r = requests.get(url, headers=headers)
        if r.status_code in (200, 404):
            try:
                return r.json()
            except ValueError:
                pass
        print("Open Food Facts respondió", r.status_code, "para", ean, "- reintento")
        time.sleep(5 * (intento + 1))
    return None


def bronze_categoria(category_id, headers):
    # 1) Categoría (si ya está descargada se reutiliza, para conservar la fecha real de descarga)
    ruta_cat = f"data/raw/mercadona/categories/{category_id}.json"
    if os.path.exists(ruta_cat):
        with open(ruta_cat, encoding="utf-8") as f:
            datos = json.load(f)
    else:
        url = f"https://tienda.mercadona.es/api/categories/{category_id}/?lang=es&wh=vlc1"
        r = requests.get(url)
        r.raise_for_status()
        datos = r.json()
        with open(ruta_cat, "w", encoding="utf-8") as f:
            json.dump(datos, f, ensure_ascii=False, indent=2)

    productos = []
    for grupo in datos["categories"]:
        productos.extend(grupo["products"])

    resultados = []
    for p in productos:
        product_id = p["id"]

        # 2) Detalle en Mercadona (si ya está descargado, se reutiliza)
        ruta = f"data/raw/mercadona/products/{product_id}.json"
        if os.path.exists(ruta):
            with open(ruta, encoding="utf-8") as f:
                detalle = json.load(f)
        else:
            r = requests.get(f"https://tienda.mercadona.es/api/products/{product_id}/?lang=es&wh=vlc1")
            if r.status_code != 200:
                print("Error en Mercadona:", product_id, r.status_code)
                continue
            detalle = r.json()
            with open(ruta, "w", encoding="utf-8") as f:
                json.dump(detalle, f, ensure_ascii=False, indent=2)
            time.sleep(1)

        # 3) Ficha en Open Food Facts (si ya está descargada, se reutiliza)
        ean = detalle.get("ean")
        encontrado = False
        off_error = False
        if ean:
            ruta_off = f"data/raw/openfoodfacts/{ean}.json"
            if os.path.exists(ruta_off):
                with open(ruta_off, encoding="utf-8") as f:
                    off = json.load(f)
            else:
                off = pedir_off(ean, headers)
                if off is not None:
                    with open(ruta_off, "w", encoding="utf-8") as f:
                        json.dump(off, f, ensure_ascii=False, indent=2)
                time.sleep(1)
            if off is None:
                off_error = True
            else:
                encontrado = off.get("status") == 1

        resultados.append({
            "categoria_id": category_id,
            "product_id": product_id,
            "nombre": detalle["display_name"],
            "ean": ean,
            "off_encontrado": encontrado,
            "off_error": off_error,
        })

    return pd.DataFrame(resultados)

### Categorías que descargo

Mantengo las 3 categorías de la entrega 2 (arroz, legumbres y pasta) y añado 40 más, hasta **43 subcategorías** de alimentos envasados. Elegí las que suelen tener código de barras y cuyos nutrientes se expresan por 100 g: cereales y galletas, aperitivos, chocolate y mermelada, charcutería y quesos, conservas, yogures y postres, huevos y mantequilla, pan envasado, salsas, pizzas y platos preparados.

Dejé fuera lo que no encaja con este análisis: las bebidas (se miden en mililitros), la fruta, la verdura, la carne y el pescado frescos (a menudo no tienen un código de barras fiable) y todo lo que no es alimentación (limpieza, cosmética, mascotas...).

Si una categoría falla, la celda lo anota en `fallos` y sigue con las demás. Al volver a ejecutarla, retoma lo que ya está descargado. Los 88 productos de la entrega 2 se descargaron el 28 de octubre de 2026 y el resto el 11 de noviembre.

In [3]:
CATEGORIAS = {
    118: "Arroz", 121: "Legumbres", 120: "Pasta y fideos",
    78: "Cereales", 80: "Galletas", 79: "Tortitas",
    135: "Aceitunas y encurtidos", 133: "Frutos secos", 132: "Patatas fritas y snacks",
    92: "Chocolate", 90: "Mermelada y miel",
    48: "Aves y jamón cocido", 52: "Bacón y salchichas", 49: "Chopped y mortadela",
    51: "Embutido curado", 50: "Jamón serrano", 58: "Paté y sobrasada",
    54: "Queso curado", 56: "Queso lonchas y rallado", 53: "Queso untable y fresco",
    43: "Embutido", 44: "Hamburguesas y picadas",
    122: "Atún y conservas de pescado", 123: "Berberechos y mejillones",
    127: "Conservas de verdura y frutas", 126: "Tomate",
    103: "Yogures desnatados", 109: "Yogures griegos", 104: "Yogures naturales y sabores",
    110: "Flan y natillas", 111: "Gelatina y otros postres",
    77: "Huevos", 75: "Mantequilla y margarina",
    60: "Pan de molde", 62: "Pan tostado y rallado", 64: "Picos y rosquilletas",
    66: "Bollería envasada", 65: "Bollería de horno",
    116: "Mayonesa, ketchup y mostaza", 117: "Otras salsas",
    138: "Pizzas", 140: "Platos preparados calientes", 142: "Platos preparados fríos",
}

dfs = []
fallos = []
for cid, nombre_cat in CATEGORIAS.items():
    try:
        d = bronze_categoria(cid, headers)
        dfs.append(d)
        print(f"{cid} {nombre_cat}: {len(d)} productos")
    except Exception as e:
        fallos.append(cid)
        print(f"FALLO en {cid} {nombre_cat}: {e}")

bronze = pd.concat(dfs, ignore_index=True)
print("Productos descargados:", len(bronze))
print("Categorías con fallo:", fallos)

118 Arroz: 14 productos
121 Legumbres: 18 productos
120 Pasta y fideos: 56 productos
78 Cereales: 43 productos
80 Galletas: 60 productos
79 Tortitas: 8 productos
135 Aceitunas y encurtidos: 41 productos
133 Frutos secos: 68 productos
132 Patatas fritas y snacks: 43 productos
92 Chocolate: 83 productos
90 Mermelada y miel: 17 productos
48 Aves y jamón cocido: 43 productos
52 Bacón y salchichas: 26 productos
49 Chopped y mortadela: 10 productos
51 Embutido curado: 48 productos
50 Jamón serrano: 29 productos
58 Paté y sobrasada: 21 productos
54 Queso curado: 43 productos
56 Queso lonchas y rallado: 28 productos
53 Queso untable y fresco: 41 productos
43 Embutido: 14 productos
44 Hamburguesas y picadas: 25 productos
122 Atún y conservas de pescado: 31 productos
123 Berberechos y mejillones: 11 productos
127 Conservas de verdura y frutas: 51 productos
126 Tomate: 18 productos
103 Yogures desnatados: 16 productos
109 Yogures griegos: 10 productos
104 Yogures naturales y sabores: 20 productos

### ¿Cuántos productos encuentro en Open Food Facts?

Antes de seguir quiero saber cuántos productos de Mercadona existen también en Open Food Facts, porque de eso depende cuántos tendrán datos nutricionales. De las 1401 filas descargadas, 1172 (el 83,7 %) aparecen y 229 no.

La cobertura cambia mucho según la categoría. Es total (100 %) en varias categorías, por ejemplo jamón cocido, pan de molde, mantequilla, tortitas y mayonesa, y baja en embutido (14 %), queso curado (16 %), bollería de horno (25 %), jamón serrano (52 %) y bollería envasada (55 %).

En la entrega 2 observé que todos los productos que faltaban eran Hacendado, pero con más categorías eso ya no se cumple: aparece el 88,9 % de los Hacendado (941 de 1059), frente al 67,5 % de los de otras marcas (231 de 342). De los 229 que faltan, 118 son Hacendado y 111 son de otras marcas. Lo dejo anotado como algo que he observado y no como una explicación: no sé por qué faltan.

Los códigos de barras que no tienen 13 dígitos los marco en la columna `ean_estandar` para tenerlos localizados.

In [4]:
cobertura = bronze.groupby("categoria_id").agg(
    productos=("product_id", "count"),
    encontrados=("off_encontrado", "sum"),
    errores_off=("off_error", "sum"),
)
cobertura["pct_encontrados"] = (cobertura["encontrados"] / cobertura["productos"] * 100).round(1)
print(cobertura)

# ¿Los que faltan son de una marca concreta?
bronze["es_hacendado"] = bronze["nombre"].str.contains("Hacendado", case=False)
print(bronze.groupby("es_hacendado")["off_encontrado"].agg(["sum", "count"]))

# Productos que no están en OFF
print(bronze[~bronze["off_encontrado"]][["categoria_id", "nombre", "ean"]])

              productos  encontrados  errores_off  pct_encontrados
categoria_id                                                      
43                   14            2            0             14.3
44                   25           23            0             92.0
48                   43           43            0            100.0
49                   10           10            0            100.0
50                   29           15            0             51.7
51                   48           33            0             68.8
52                   26           20            0             76.9
53                   41           37            0             90.2
54                   43            7            0             16.3
56                   28           21            0             75.0
58                   21           15            0             71.4
60                   40           40            0            100.0
62                   30           25            0             

## 2. Silver: limpiar y unir los datos

En esta fase preparo los datos para poder usarlos:
1. Elimino los productos repetidos (un mismo producto puede aparecer en dos categorías).
2. Saco de las fichas de Open Food Facts los valores nutricionales por cada 100 g: calorías, proteína, grasa, hidratos, y fibra, azúcares y sal cuando están.
3. Marco qué productos tienen completos los cuatro datos principales (calorías, proteína, grasa e hidratos).

No borro los productos que tienen datos incompletos. Los dejo marcados para poder contar después cuántos se pierden y por qué.

In [5]:
antes = len(bronze)
silver = bronze.drop_duplicates(subset="product_id").copy()
print("Filas antes:", antes, "| después:", len(silver))

campos = ["energy-kcal_100g", "proteins_100g", "fat_100g", "carbohydrates_100g"]
extras = ["fiber_100g", "sugars_100g", "salt_100g"]

filas = []
for _, fila in silver[silver["off_encontrado"]].iterrows():
    with open(f"data/raw/openfoodfacts/{fila['ean']}.json", encoding="utf-8") as f:
        off = json.load(f)
    n = off["product"].get("nutriments", {})
    registro = {"product_id": fila["product_id"]}
    for c in campos + extras:
        registro[c] = n.get(c)
    filas.append(registro)

silver = silver.merge(pd.DataFrame(filas), on="product_id", how="left")

silver["macros_completos"] = silver[campos].notna().all(axis=1)
silver["ean_estandar"] = silver["ean"].astype(str).str.len() == 13

silver["macros_completos"].value_counts()

Filas antes: 1401 | después: 1373


macros_completos
True     1104
False     269
Name: count, dtype: int64

### ¿Son fiables los valores nutricionales?

Open Food Facts es una base de datos que rellena cualquier persona, no una fuente oficial. Eso la hace muy útil, porque tiene muchísimos productos, pero también significa que alguna ficha puede tener un dato mal introducido. Por eso compruebo los datos antes de usarlos.

La primera comprobación es esta: las calorías de un alimento se pueden calcular a partir de sus nutrientes (4 calorías por gramo de proteína, 4 por gramo de hidratos, 9 por gramo de grasa y 2 por gramo de fibra). Comparo ese cálculo con las calorías que declara la ficha. Si se parecen, el dato es coherente.

Distingo dos casos de fibra, porque no son lo mismo:
- Fibra informada (incluido el valor 0): sé cuánta fibra hay, así que la incluyo en el cálculo y la diferencia debe quedar dentro del 10 %. Si no, el producto es incoherente.
- Fibra ausente (la ficha no dice nada): no sé si hay fibra, y la fibra solo puede sumar calorías, nunca restarlas. Si las calorías declaradas son entre un 10 % y un 25 % más altas que el cálculo, el producto queda como no_concluyente, porque podría explicarse por la fibra que falta.

Los cuatro estados de calidad son:
- coherente: el cálculo y las calorías declaradas coinciden.
- no_concluyente: fibra ausente y calorías algo más altas de lo calculado.
- incoherente: la diferencia es demasiado grande para explicarla.
- sin_datos: faltan valores nutricionales.

Que las calorías cuadren no demuestra que el resto de nutrientes estén bien, así que más abajo añado controles de rangos.

Para saber si la comprobación acierta, miré a mano las etiquetas de dos productos marcados como incoherentes (el arroz cocido redondo Sabroz y la quinoa cocida Sabroz). En los dos casos el dato de Open Food Facts estaba mal.

### Correcciones manuales

Al revisar los productos incoherentes miré en la web de Mercadona la etiqueta de la lenteja cocida Hacendado de 295 g (EAN 8480000053329). La etiqueta no declara fibra, pero Open Food Facts tenía la fibra como 0. Un cero y un dato ausente no son lo mismo, así que la dejo vacía.

Hay otro tarro de la misma lenteja (570 g, EAN 8480000260307) cuya etiqueta tampoco declara fibra. En ese caso Open Food Facts ya tenía la fibra vacía, así que no hubo que corregir nada.

Las correcciones manuales están en data/manual/correcciones_manuales.csv, con el motivo y la fuente de cada una. Los datos originales no se tocan: siguen igual en Bronze, y la columna correccion_manual marca en Silver qué productos se han corregido.

In [6]:
correcciones = pd.read_csv("data/manual/correcciones_manuales.csv", dtype={"ean": str})

for _, c in correcciones.iterrows():
    filas = silver["ean"].astype(str) == c["ean"]
    silver.loc[filas, c["columna"]] = c["valor_corregido"]  # vacío = dato ausente

silver["correccion_manual"] = silver["ean"].astype(str).isin(correcciones["ean"])
print("Productos corregidos:", silver["correccion_manual"].sum())

Productos corregidos: 1


In [7]:
silver["kcal_calculadas"] = (
    4 * silver["proteins_100g"] + 4 * silver["carbohydrates_100g"] + 9 * silver["fat_100g"]
)
silver["diferencia_pct"] = (
    (silver["energy-kcal_100g"] - silver["kcal_calculadas"])
    / silver["kcal_calculadas"].where(silver["kcal_calculadas"] > 0) * 100
).round(1)


def estado_coherencia(fila):
    if not fila["macros_completos"]:
        return "sin_datos"
    calc = fila["kcal_calculadas"]
    kcal = fila["energy-kcal_100g"]
    fibra = fila["fiber_100g"]

    # Si los macros suman 0 g no se puede calcular una diferencia: solo es coherente si declara 0 kcal
    if calc == 0:
        return "coherente" if kcal == 0 else "incoherente"

    # Fibra informada (incluida la fibra 0): sabemos cuánta hay, así que la usamos y decidimos
    if pd.notna(fibra):
        esperado = calc + 2 * fibra
        dif = (kcal - esperado) / esperado * 100
        return "coherente" if abs(dif) <= 10 else "incoherente"

    # Fibra ausente (no sabemos si hay): solo puede añadir kcal, nunca quitarlas
    dif = (kcal - calc) / calc * 100
    if abs(dif) <= 10:
        return "coherente"
    if 10 < dif <= 25:
        return "no_concluyente"
    return "incoherente"


silver["estado_calidad"] = silver.apply(estado_coherencia, axis=1)
silver["estado_calidad"].value_counts()


estado_calidad
coherente         1024
sin_datos          269
incoherente         59
no_concluyente      21
Name: count, dtype: int64

### Precio, tamaño y formato

El precio, el tamaño del envase y el formato (paquete, pack, etc.) vienen de la ficha del producto en Mercadona. En algunos productos el formato no viene indicado, y lo dejo vacío en lugar de inventarlo.

Mercadona da un precio de referencia, pero no siempre por kilo. En esta descarga, 1311 productos lo tienen por kilo, 41 por litro, 2 por 100 g y 19 por unidad, decena o docena. Para poder comparar productos creo una columna `precio_kg` (euros por kilo):
- Por kilo: lo dejo igual.
- Por 100 g: lo multiplico por 10.
- Por litro: lo dejo igual, suponiendo que 1 litro pesa aproximadamente 1 kilo. Es una aproximación.
- Por unidad, decena o docena: no se puede pasar a kilos sin saber cuánto pesa cada unidad, así que queda vacío y esos productos no tendrán cálculos por euro.

In [8]:
filas_precio = []
for pid in silver["product_id"]:
    with open(f"data/raw/mercadona/products/{pid}.json", encoding="utf-8") as f:
        d = json.load(f)
    pi = d["price_instructions"]
    filas_precio.append({
        "product_id": pid,
        "precio_unidad": float(pi["unit_price"]),
        "precio_referencia": float(pi["reference_price"]),
        "unidad_referencia": pi["reference_format"],
        "tamano_envase": pi["unit_size"],
        "unidad_tamano": pi["size_format"],
        "formato": d.get("packaging"),
    })

silver = silver.merge(pd.DataFrame(filas_precio), on="product_id", how="left")
print(silver["unidad_referencia"].value_counts())

# Precio por kilo comparable (kg igual, 100 g x 10, litro ~ kilo; unidades, decenas y docenas quedan vacíos)
silver["precio_kg"] = np.select(
    [silver["unidad_referencia"].isin(["kg", "L"]), silver["unidad_referencia"] == "100 g"],
    [silver["precio_referencia"], silver["precio_referencia"] * 10],
    default=np.nan,
)
print("\nProductos con precio por kilo comparable:", silver["precio_kg"].notna().sum(), "de", len(silver))

unidad_referencia
kg       1311
L          41
ud         10
dc          8
100 g       2
dz          1
Name: count, dtype: int64

Productos con precio por kilo comparable: 1354 de 1373


### Controles de rangos y precios

Compruebo cada producto con reglas de sentido común:
- Las calorías deben estar entre 0 y 900 por cada 100 g, y los nutrientes entre 0 y 100 g.
- Proteína, grasa, hidratos y fibra no pueden sumar más de 100 g por cada 100 g de producto (dejo 1 g de margen por redondeos).
- Los azúcares forman parte de los hidratos, así que no pueden superarlos (margen de 0,5 g).
- El precio del producto y el precio de referencia deben ser mayores que cero, para no dividir por cero ni calcular precios imposibles.

Los productos que no pasan algún control no se borran: quedan marcados en la columna problemas_rango y no entran en el Gold.

In [9]:
macros_g = ["proteins_100g", "fat_100g", "carbohydrates_100g", "fiber_100g", "sugars_100g", "salt_100g"]


def problemas_rango(fila):
    problemas = []

    kcal = fila["energy-kcal_100g"]
    if pd.notna(kcal) and not (0 <= kcal <= 900):
        problemas.append("kcal_fuera_de_rango")

    for c in macros_g:
        v = fila[c]
        if pd.notna(v) and not (0 <= v <= 100):
            problemas.append(c + "_fuera_de_rango")

    # Los nutrientes que pesan no pueden sumar más de 100 g por cada 100 g de producto
    pesos = [fila[c] for c in ["proteins_100g", "fat_100g", "carbohydrates_100g", "fiber_100g"] if pd.notna(fila[c])]
    if sum(pesos) > 101:
        problemas.append("suma_nutrientes_mayor_100g")

    # Los azúcares forman parte de los hidratos
    if pd.notna(fila["sugars_100g"]) and pd.notna(fila["carbohydrates_100g"]):
        if fila["sugars_100g"] > fila["carbohydrates_100g"] + 0.5:
            problemas.append("azucares_mayor_hidratos")

    # Precios positivos (evita divisiones por cero más adelante)
    if not (fila["precio_unidad"] > 0 and fila["precio_referencia"] > 0):
        problemas.append("precio_no_positivo")

    return ";".join(problemas)


silver["problemas_rango"] = silver.apply(problemas_rango, axis=1)
silver["rango_ok"] = silver["problemas_rango"] == ""

print(silver["rango_ok"].value_counts())
silver[~silver["rango_ok"]][["nombre", "problemas_rango"]]

rango_ok
True     1364
False       9
Name: count, dtype: int64


,nombre,problemas_rango
111,Cereales con salvado de trigo Fibra Sticks Hac...,suma_nutrientes_mayor_100g
131,Rosquillas de mantequilla Hacendado,suma_nutrientes_mayor_100g
157,Galletas rebuenas Hacendado rellenas sabor cho...,suma_nutrientes_mayor_100g
342,Snack de maíz Redes sabor barbacoa Hacendado,suma_nutrientes_mayor_100g
423,Bombones de pistacho Hacendado,suma_nutrientes_mayor_100g
680,Queso lonchas sin lactosa de vaca Hacendado,azucares_mayor_hidratos
946,"Gelatina sabores mango maracuyá, guayaba y piñ...",azucares_mayor_hidratos
1045,Pan tostado 100% espelta integral Hacendado,kcal_fuera_de_rango
1064,Pan de fibra y sésamo Hacendado,suma_nutrientes_mayor_100g


### Fecha real de cada descarga

Guardo para cada producto la fecha en la que se descargaron sus archivos. Uso la fecha de modificación de cada archivo en data/raw: como el notebook no reescribe los archivos que ya existen, esa fecha sigue siendo la del día de la descarga aunque se reutilice la caché.

Aquí se ven las dos descargas: 88 productos del 28 de octubre de 2026 (los de la entrega 2: arroz, legumbres y pasta) y 1285 del 11 de noviembre de 2026 (las categorías nuevas). Mercadona cambia su catálogo con el tiempo, así que es importante saber de qué día es cada dato.

In [10]:
def fecha_archivo(ruta):
    if not os.path.exists(ruta):
        return None
    return datetime.datetime.fromtimestamp(os.path.getmtime(ruta)).strftime("%Y-%m-%d")


silver["fecha_descarga_mercadona"] = silver["product_id"].apply(
    lambda pid: fecha_archivo(f"data/raw/mercadona/products/{pid}.json")
)
silver["fecha_descarga_off"] = silver["ean"].apply(
    lambda ean: fecha_archivo(f"data/raw/openfoodfacts/{ean}.json")
)

print(silver["fecha_descarga_mercadona"].value_counts())
print(silver["fecha_descarga_off"].value_counts())

fecha_descarga_mercadona
2026-11-11    1285
2026-10-28      88
Name: count, dtype: int64
fecha_descarga_off
2026-11-11    1285
2026-10-28      88
Name: count, dtype: int64


### ¿Está el producto seco, cocinado o fresco? ¿De qué tipo es?

**Estado.** Esto importa en arroz, legumbres y pasta, porque no se pueden comparar bien un arroz seco y un arroz ya cocinado. El cocinado lleva agua, así que tiene menos calorías y menos proteína en cada 100 g, y además cuesta más por kilo. Si los mezclo, una cuenta como "proteína por euro" daría resultados engañosos. En el resto de categorías esta distinción no tiene sentido, y el estado vale `no_aplica`.

Averiguo el estado a partir del nombre del producto:
- seco: es el valor por defecto.
- cocinado: el nombre contiene "cocid" pero no "precocid", porque las placas de lasaña precocidas se venden secas.
- fresco: el nombre contiene "fresca" (pasta fresca, gnocchi...).
- instantaneo: fideos orientales y ramen. Tienen entre 93 y 204 calorías por 100 g, que son muy pocas para unos fideos instantáneos secos. Lo más probable es que los datos se refieran al plato ya preparado, aunque no lo he podido confirmar. Por eso este grupo queda aparte y no se compara con los demás.

Para comprobar que el reparto tiene sentido miro las calorías de cada grupo: los cocinados quedan por debajo de unas 165 calorías, los frescos entre 214 y 306, y los secos entre 292 y 394 (frescos y secos se solapan un poco). Clasificar por el nombre es un método sencillo y puede fallar con productos nuevos.

**Grupo.** Para poder analizar los resultados por tipo de producto, agrupo las categorías en cuatro: `base` (arroz, legumbres, pasta, huevos, lácteos, quesos, embutido, conservas, pan, frutos secos...), `snack_dulce` (galletas, snacks, chocolate, bollería, postres), `salsa` y `plato_preparado` (pizzas y platos listos).

En la tabla de calorías por estado se ve un máximo de 5232 kcal por 100 g en `no_aplica`. Es un valor imposible de Open Food Facts: el control de rangos lo deja marcado y no entra en el Gold.

In [11]:
nombre = silver["nombre"].str.lower()
condiciones = [
    nombre.str.contains("orientales|ramen"),
    nombre.str.contains("cocid") & ~nombre.str.contains("precocid"),
    nombre.str.contains("fresca"),
]
silver["estado"] = np.select(condiciones, ["instantaneo", "cocinado", "fresco"], default="seco")

# El estado seco/cocinado solo tiene sentido en arroz, legumbres y pasta
silver.loc[~silver["categoria_id"].astype(int).isin([118, 121, 120]), "estado"] = "no_aplica"

# Grupo de producto, para presentar resultados por tipo
grupos = {
    "base": [118, 121, 120, 77, 75, 103, 109, 104, 54, 56, 53, 48, 49, 51, 50, 52, 58, 43, 44,
             122, 123, 127, 126, 60, 62, 64, 133, 135, 78],
    "snack_dulce": [80, 79, 132, 92, 90, 110, 111, 66, 65],
    "salsa": [116, 117],
    "plato_preparado": [138, 140, 142],
}
mapa_grupo = {cid: g for g, ids in grupos.items() for cid in ids}
silver["grupo"] = silver["categoria_id"].astype(int).map(mapa_grupo)
print(silver["grupo"].value_counts(dropna=False))

print(silver.groupby("estado")["energy-kcal_100g"].agg(["count", "min", "max"]))

silver.to_csv("data/silver/productos_silver_ampliado.csv", index=False)
silver.shape

grupo
base               851
snack_dulce        346
plato_preparado    129
salsa               47
Name: count, dtype: int64
             count     min          max
estado                                 
cocinado        11   83.00   164.000000
fresco           9  214.00   306.000000
instantaneo      4   93.00   204.000000
no_aplica     1036    0.29  5232.876712
seco            48  292.00   394.000000


(1373, 33)

## 3. Gold: la tabla final

La tabla final tiene una fila por cada producto de Mercadona que se puede usar. Entran los productos que cumplen las dos condiciones siguientes:
- Sus calorías han pasado la comprobación: los marcados como coherente y los no_concluyente.
- Han superado los controles de rangos y precios.

La columna estado_calidad se queda en la tabla para que, quien la use, pueda decidir si quiere ser más estricto y quedarse solo con los coherentes. Los productos incoherentes o sin datos no entran, pero siguen en Silver para poder saber qué se descartó.

**Qué ha pasado con los 1373 productos distintos de la Silver:**

| Estado de calidad | Productos |
|---|---|
| Coherente | 1024 |
| No concluyente | 21 |
| Incoherente | 59 |
| Sin datos | 269 |

Nueve productos no pasan el control de rango, y ocho de ellos tenían calorías coherentes o no concluyentes. Por eso el Gold final tiene 1037 productos y 22 columnas (1045 − 8): 625 del grupo base, 259 snacks y dulces, 107 platos preparados y 46 salsas.

Añado tres columnas calculadas, pensadas para el objetivo del proyecto:
- proteina_g_por_euro: cuántos gramos de proteína se obtienen por cada euro.
- kcal_por_euro: cuántas calorías se obtienen por cada euro.
- pct_kcal_proteina: qué porcentaje de las calorías del producto viene de la proteína.

Las dos primeras usan el precio por kilo (`precio_kg`), así que quedan vacías en los productos cuyo precio es por unidad, decena o docena. En arroz, legumbres y pasta solo se pueden comparar productos en el mismo estado (seco con seco, cocinado con cocinado).

Los productos de estado instantaneo (fideos orientales y ramen, 4 productos) siguen en la tabla, pero con las columnas por euro vacías. No he podido confirmar si su información nutricional se refiere al producto seco o al plato ya preparado, y hasta resolverlo no se deben usar para comparar precio y nutrición.

In [12]:
# Gold ampliado: calorías coherentes o no concluyentes, y controles de rango superados
gold = silver[
    silver["estado_calidad"].isin(["coherente", "no_concluyente"]) & silver["rango_ok"]
].copy()

gold = gold.rename(columns={
    "energy-kcal_100g": "kcal_100g",
    "proteins_100g": "proteina_100g",
    "fat_100g": "grasa_100g",
    "carbohydrates_100g": "hidratos_100g",
    "fiber_100g": "fibra_100g",
    "sugars_100g": "azucares_100g",
    "salt_100g": "sal_100g",
})

# Los cálculos por euro usan el precio por kilo comparable (vacío si el precio es por unidad o docena)
precio_ok = gold["precio_kg"].where(gold["precio_kg"] > 0)
kcal_ok = gold["kcal_100g"].where(gold["kcal_100g"] > 0)

gold["proteina_g_por_euro"] = (gold["proteina_100g"] * 10 / precio_ok).round(1)
gold["kcal_por_euro"] = (gold["kcal_100g"] * 10 / precio_ok).round(0)
gold["pct_kcal_proteina"] = (gold["proteina_100g"] * 4 / kcal_ok * 100).round(1)

instantaneos = gold["estado"] == "instantaneo"
gold.loc[instantaneos, ["proteina_g_por_euro", "kcal_por_euro"]] = np.nan

columnas = [
    "product_id", "ean", "nombre", "categoria_id", "grupo", "formato", "estado",
    "precio_unidad", "precio_referencia", "precio_kg", "tamano_envase",
    "kcal_100g", "proteina_100g", "grasa_100g", "hidratos_100g",
    "fibra_100g", "azucares_100g", "sal_100g",
    "proteina_g_por_euro", "kcal_por_euro", "pct_kcal_proteina",
    "estado_calidad",
]
gold = gold[columnas]

gold.to_csv("data/gold/productos_gold_ampliado.csv", index=False)
print(gold.shape)
print(gold["grupo"].value_counts())
print(gold["estado"].value_counts())
print("Sin cálculo por euro (precio no por kilo o instantáneo):", gold["proteina_g_por_euro"].isna().sum())
gold.sort_values("proteina_g_por_euro", ascending=False).head(10)

(1037, 22)
grupo
base               625
snack_dulce        259
plato_preparado    107
salsa               46
Name: count, dtype: int64
estado
no_aplica      967
seco            48
cocinado         9
fresco           9
instantaneo      4
Name: count, dtype: int64
Sin cálculo por euro (precio no por kilo o instantáneo): 11


,product_id,ean,nombre,categoria_id,grupo,formato,estado,precio_unidad,precio_referencia,precio_kg,...,proteina_100g,grasa_100g,hidratos_100g,fibra_100g,azucares_100g,sal_100g,proteina_g_por_euro,kcal_por_euro,pct_kcal_proteina,estado_calidad
28,5330,8480000053305,Lenteja pardina Hacendado,121,base,Paquete,seco,1.85,1.850,1.850,...,24.50,1.20,48.10,9.6,1.3,0.05,132.4,1730.0,30.6,coherente
23,5124,8480000051240,Alubia blanca Hacendado,121,base,Paquete,seco,2.45,2.450,2.450,...,21.40,1.00,40.00,18.9,2.3,0.03,87.3,1192.0,29.3,coherente
17,5214,8480000052148,Garbanzo Hacendado,121,base,Paquete,seco,2.15,2.150,2.150,...,18.60,6.10,47.50,14.4,6.0,0.10,86.5,1619.0,21.4,coherente
99,86341,4008713713411,Copos de avena Brüggen,78,base,Caja,no_aplica,1.30,1.625,1.625,...,14.00,7.00,59.00,10.0,0.7,0.02,86.2,2308.0,14.9,coherente
24,67608,8480000676085,Alubia negra Hacendado,121,base,Paquete,seco,2.50,2.500,2.500,...,21.40,1.00,40.00,18.9,2.3,0.03,85.6,1168.0,29.3,coherente
1058,82219,8480000822192,Pan rallado Hacendado,62,base,Paquete,no_aplica,0.85,1.134,1.134,...,9.58,1.26,78.12,NaN,3.4,1.00,84.5,3254.0,10.4,coherente
27,23453,8480000234537,Alubia pinta Hacendado,121,base,Paquete,seco,2.30,2.300,2.300,...,19.00,1.40,52.50,25.4,2.2,0.03,82.6,1517.0,21.8,coherente
42,6344,8402001015717,Hélices con vegetales Hacendado,120,base,Paquete,seco,1.60,1.600,1.600,...,13.00,1.50,76.00,4.0,0.0,0.00,81.2,2262.0,14.4,coherente
35,6363,8480000063632,Pasta estrellas Hacendado,120,base,Paquete,seco,0.85,1.700,1.700,...,13.00,1.50,72.00,3.5,3.6,0.01,76.5,2124.0,14.4,coherente
36,6258,8480000062581,Pasta maravilla Hacendado,120,base,Paquete,seco,0.85,1.700,1.700,...,13.00,1.50,72.00,3.5,3.6,0.01,76.5,2124.0,14.4,coherente


### Quién se queda fuera del Gold

Antes de pasar al experimento de Machine Learning miro qué productos no entran en el Gold y si hay algo raro en los que sí entran:
- el estado de calidad de cada grupo,
- cuántos productos no superan el control de rango,
- qué parte de los productos sin datos son Hacendado (para ver si los que faltan se parecen a los que sí tengo),
- y que todos los productos del Gold tienen la proteína informada, que es lo que quiero estimar.

De los 1373 productos distintos, 269 no tienen datos nutricionales completos en Open Food Facts. El 55,8 % de ellos son Hacendado, frente al 80,6 % del resto. Es decir, los productos sin datos tienen más variedad de marcas que los que sí tengo. Es un sesgo que tendré en cuenta en el experimento.

In [13]:
print("Estado de calidad por grupo:")
print(pd.crosstab(silver["grupo"], silver["estado_calidad"], margins=True))

print("\nControl de rango:")
print(silver["rango_ok"].value_counts())

hacendado = silver["nombre"].str.contains("hacendado", case=False, na=False)
print("\n% Hacendado entre sin_datos:", round(hacendado[silver["estado_calidad"] == "sin_datos"].mean() * 100, 1))
print("% Hacendado entre el resto:   ", round(hacendado[silver["estado_calidad"] != "sin_datos"].mean() * 100, 1))
print("\nProteína informada en el Gold:", gold["proteina_100g"].notna().sum(), "de", len(gold))
print(gold.groupby("grupo")["proteina_100g"].describe().round(1))

Estado de calidad por grupo:
estado_calidad   coherente  incoherente  no_concluyente  sin_datos   All
grupo                                                                   
base                   612           47              16        176   851
plato_preparado        107            4               0         18   129
salsa                   43            0               3          1    47
snack_dulce            262            8               2         74   346
All                   1024           59              21        269  1373

Control de rango:
rango_ok
True     1364
False       9
Name: count, dtype: int64

% Hacendado entre sin_datos: 55.8
% Hacendado entre el resto:    80.6

Proteína informada en el Gold: 1037 de 1037
                 count  mean  std  min  25%   50%   75%   max
grupo                                                        
base             625.0  12.2  8.6  0.0  5.2  11.5  17.0  50.0
plato_preparado  107.0   8.9  4.6  0.6  5.6   8.8  12.5  21.0
salsa         

## Resultado y siguiente paso

Este notebook deja dos archivos, que no se publican en el repositorio:
- `data/silver/productos_silver_ampliado.csv`: 1373 productos únicos con todos los controles y marcas de calidad.
- `data/gold/productos_gold_ampliado.csv`: 1037 productos listos para usar.

Los 269 productos sin datos nutricionales se quedan en Silver. Estimar su proteína es el objetivo de `experimento_ml.ipynb`.